# Ensemble Translation Inference — ByT5 + QLoRA Qwen + mBART-50

Loads three independently-trained translation checkpoints, generates a candidate translation from each, and picks a final answer per sentence using a **Minimum Bayes Risk (MBR) consensus orchestrator**.

**`EXECUTION_MODE`** (next cell): `"online"` installs packages, logs in to Hugging Face, and sets up MLflow tracking; pulls mBART-50 and Qwen from the Hub. `"offline"` skips all of that — every model loads from local Kaggle input paths, no network access.

**ByT5** is always local (Kaggle Model only, no Hub repo), regardless of mode.

Source text is used as-is from `train.csv`/`test.csv`, no normalization step.

Models are loaded, run, and fully unloaded one at a time to keep peak GPU memory at "one model" instead of three.

In [12]:
EXECUTION_MODE = "offline"  # "online" or "offline"
assert EXECUTION_MODE in ("online", "offline")
IS_ONLINE = EXECUTION_MODE == "online"
print(f"Running in {EXECUTION_MODE.upper()} mode")


Running in OFFLINE mode


In [13]:
SEED = 42

import random
import numpy as np
import torch
from transformers import set_seed as _hf_set_seed

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    _hf_set_seed(seed)

seed_everything()


In [14]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [15]:
import os
from kaggle_secrets import UserSecretsClient

_secrets = UserSecretsClient()

# Local Kaggle input paths (offline mode) — dedicated weights account, safe to hardcode.
# Phase 2: all three members retrained on the shared split (seed 42) and published by their notebooks
# with PUBLISH_MODELS = True. Check the exact paths (version number) in the Input panel after attaching.
MBART_OFFLINE_PATH = "/kaggle/input/models/eeee13/akkadian-mbart/pytorch/mbart50-seed42/1"
QWEN_BASE_OFFLINE_PATH = "/kaggle/input/models/eeee13/qwen2.5-7b-instruct/pytorch/base-v1/1"
QWEN_ADAPTER_OFFLINE_PATH = "/kaggle/input/models/eeee13/akkadika/pytorch/qwen-lora-seed42/1"
BYT5_REPO = "/kaggle/input/models/eeee13/akkadian-byt5/pytorch/byt5-seed42/1"

# Hub repo ids + credentials — kept in Secrets.
HF_TOKEN = _secrets.get_secret("HF_TOKEN") if IS_ONLINE else None
MLFLOW_USERNAME = _secrets.get_secret("MLFLOW_USERNAME") if IS_ONLINE else None
MLFLOW_PASSWORD = _secrets.get_secret("MLFLOW_PASSWORD") if IS_ONLINE else None
MBART_REPO = _secrets.get_secret("HF_REPO_MBART") if IS_ONLINE else None
QWEN_BASE_REPO = "Qwen/Qwen2.5-7B-Instruct" if IS_ONLINE else None
QWEN_ADAPTER_REPO = _secrets.get_secret("HF_REPO_QWEN_ADAPTER") if IS_ONLINE else None

if IS_ONLINE:
    for _name, _val in [("MBART_REPO", MBART_REPO), ("QWEN_ADAPTER_REPO", QWEN_ADAPTER_REPO)]:
        assert _val, f"{_name} is empty — check Add-ons > Secrets."
else:
    for _name, _val in [("MBART_OFFLINE_PATH", MBART_OFFLINE_PATH),
                         ("QWEN_BASE_OFFLINE_PATH", QWEN_BASE_OFFLINE_PATH),
                         ("QWEN_ADAPTER_OFFLINE_PATH", QWEN_ADAPTER_OFFLINE_PATH)]:
        assert _val, f"{_name} is empty — fill in the local Kaggle input path above."

In [16]:
import sys
import subprocess

if IS_ONLINE:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "transformers", "accelerate", "peft", "bitsandbytes", "sentencepiece", "mlflow"],
        check=True,
    )
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "dagshub", "--no-deps"], check=True)

    if HF_TOKEN:
        from huggingface_hub import login
        login(token=HF_TOKEN)
    else:
        print("WARNING: HF_TOKEN secret is empty — private Hub repo pulls will fail.")
else:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--no-index",
         "--find-links=/kaggle/input/datasets/eeee13/bitsandbytes", "bitsandbytes"],
        check=True,
    )
    print("Offline mode: installed bitsandbytes from local wheel, skipping HF login.")


Offline mode: installed bitsandbytes from local wheel, skipping HF login.


## MLflow tracking

Logged once per model plus once for the ensemble, on the validation split:

- `val_bleu`, `val_chrf_pp` — corpus BLEU / chrF++ against the validation references
- `val_geo_mean_chrf_bleu` — geometric mean of the two
- `val_avg_logprob` — the mean beam-search sequence log-probability `generate()` returned, used here as a rough **stand-in for a validation loss** (it's not a true loss — these are frozen, already-trained checkpoints, there's no training loop here to compute one from — but it's the only per-sentence confidence signal available at inference time and moves in the same direction as one)

Each model gets its own nested MLflow run under a parent `ensemble-inference` run, so you can compare them side by side in the MLflow/DagsHub UI the same way you compare training runs.

In [17]:
if IS_ONLINE:
    import os
    import mlflow

    os.environ["MLFLOW_TRACKING_USERNAME"] = MLFLOW_USERNAME
    os.environ["MLFLOW_TRACKING_PASSWORD"] = MLFLOW_PASSWORD

    MLFLOW_TRACKING_URI = _secrets.get_secret("MLFLOW_TRACKING_URI")
    assert MLFLOW_TRACKING_URI, "MLFLOW_TRACKING_URI is empty — check Add-ons > Secrets."

    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    mlflow.set_experiment("deep-past-initiative-machine-translation")
    print("MLflow tracking configured.")
else:
    print("Offline mode: skipping MLflow setup — validation metrics will still print to stdout.")


Offline mode: skipping MLflow setup — validation metrics will still print to stdout.


## Shared split, metric and run record (Phase 2)

The next four cells are identical in all four notebooks (copied from `common/` by `tools/sync_shared_cells.py`). They give every model **the same 156 validation documents** (checked by a SHA-256 of the sorted ids), **the same metric** (sacreBLEU-identical BLEU and chrF++ with word n-grams, `tests/test_metrics.py`), write one prediction file plus one JSON run record per model into `results/`, and record the package versions and GPU (`results/env_*.json`).

In [ ]:
# >>> SHARED CELL: common/split.py
# Auto-copied from the repository by tools/sync_shared_cells.py. Do not edit here;
# identical in mbart50, byte-level-seq2seq, pipeline_QwenQLoRa and ensemble-pipeline.
import hashlib
import random

ID_COL = "oare_id"
VAL_FRACTION = 0.1
SPLIT_SEED = 42
# sha256 of "\n".join(sorted(val_ids)) for the competition train.csv (1561 documents -> 156 val).
EXPECTED_VAL_SHA256 = "1939161cfcb9ff85082f20b36be2362da429f1da14f8a94f289249ffb78e1304"
EXPECTED_N_VAL = 156


def val_ids_sha256(val_ids):
    return hashlib.sha256("\n".join(sorted(map(str, val_ids))).encode("utf-8")).hexdigest()


def make_split(df, id_col=ID_COL, val_fraction=VAL_FRACTION, seed=SPLIT_SEED, check=True):
    """Return (train_df, val_df), split by document id. Rows with empty text are dropped first."""
    df = df.dropna(subset=["transliteration", "translation"]).reset_index(drop=True)
    doc_ids = sorted(df[id_col].astype(str).unique().tolist())
    random.Random(seed).shuffle(doc_ids)
    n_val = max(1, int(len(doc_ids) * val_fraction))
    val_ids = set(doc_ids[:n_val])
    is_val = df[id_col].astype(str).isin(val_ids)
    train_df = df[~is_val].reset_index(drop=True)
    val_df = df[is_val].reset_index(drop=True)
    digest = val_ids_sha256(val_ids)
    print(f"[shared split] train={len(train_df)} rows, val={len(val_df)} rows "
          f"({n_val} documents), val_ids_sha256={digest[:16]}...")
    if check:
        assert n_val == EXPECTED_N_VAL and digest == EXPECTED_VAL_SHA256, (
            "Validation split differs from the shared reference split "
            f"(n_val={n_val}, sha256={digest}). Is train.csv the original competition file?")
    assert not set(train_df[id_col].astype(str)) & val_ids, "document leakage between train and val"
    return train_df, val_df

In [ ]:
# >>> SHARED CELL: common/metrics.py
# Auto-copied from the repository by tools/sync_shared_cells.py. Do not edit here;
# identical in mbart50, byte-level-seq2seq, pipeline_QwenQLoRa and ensemble-pipeline.
import math
import re
from collections import Counter

# ---------------------------------------------------------------- BLEU (13a)
_TOK_13A_RULES = [
    (re.compile(r'([\{-\~\[-\` -\&\(-\+\:-\@\/])'), r' \1 '),
    (re.compile(r'([^0-9])([\.,])'), r'\1 \2 '),
    (re.compile(r'([\.,])([^0-9])'), r' \1 \2'),
    (re.compile(r'([0-9])(-)'), r'\1 \2 '),
]


def _tokenize_13a(line):
    line = line.replace('<skipped>', '').replace('-\n', '').replace('\n', ' ')
    if '&' in line:
        line = (line.replace('&quot;', '"').replace('&amp;', '&')
                    .replace('&lt;', '<').replace('&gt;', '>'))
    line = f' {line} '
    for pattern, repl in _TOK_13A_RULES:
        line = pattern.sub(repl, line)
    return ' '.join(line.split())


def _word_ngrams(tokens, max_order):
    return Counter(tuple(tokens[i:i + n])
                   for n in range(1, max_order + 1)
                   for i in range(len(tokens) - n + 1))


def bleu_stats(hyp, ref, max_order=4):
    """Sufficient statistics of one segment: [hyp_len, ref_len, correct_1..4, total_1..4]."""
    h = _tokenize_13a(hyp.rstrip()).split()
    r = _tokenize_13a(ref.rstrip()).split()
    h_ng, r_ng = _word_ngrams(h, max_order), _word_ngrams(r, max_order)
    correct, total = [0] * max_order, [0] * max_order
    for ng, c in h_ng.items():
        total[len(ng) - 1] += c
        if ng in r_ng:
            correct[len(ng) - 1] += min(c, r_ng[ng])
    return [len(h), len(r)] + correct + total


def bleu_from_stats(stats, max_order=4):
    sys_len, ref_len = stats[0], stats[1]
    correct, total = stats[2:2 + max_order], stats[2 + max_order:]
    bp = 1.0
    if sys_len < ref_len:
        bp = math.exp(1 - ref_len / sys_len) if sys_len > 0 else 0.0
    if not any(correct):
        return 0.0
    precisions = [0.0] * max_order
    smooth = 1.0
    for n in range(max_order):
        if total[n] == 0:
            break
        if correct[n] == 0:
            smooth *= 2
            precisions[n] = 100.0 / (smooth * total[n])
        else:
            precisions[n] = 100.0 * correct[n] / total[n]
    log_sum = sum(-9999999999 if p == 0 else math.log(p) for p in precisions)
    return bp * math.exp(log_sum / max_order)


# ---------------------------------------------------------------- chrF++
_PUNCTS = set('!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~')


def _chrf_words(sent):
    out = []
    for w in sent.split():
        if len(w) == 1:
            out.append(w)
        elif w[-1] in _PUNCTS:
            out += [w[:-1], w[-1]]
        elif w[0] in _PUNCTS:
            out += [w[0], w[1:]]
        else:
            out.append(w)
    return out


def _chrf_ngrams(sent, char_order, word_order):
    chars = ''.join(sent.split())
    grams = [Counter(chars[i:i + n] for i in range(len(chars) - n + 1))
             for n in range(1, char_order + 1)]
    words = _chrf_words(sent)
    grams += [Counter(' '.join(words[i:i + n]) for i in range(len(words) - n + 1))
              for n in range(1, word_order + 1)]
    return grams


def chrf_stats(hyp, ref, char_order=6, word_order=2):
    """Flattened [hyp, ref, match] counts for every char then word n-gram order."""
    stats = []
    for h, r in zip(_chrf_ngrams(hyp, char_order, word_order),
                    _chrf_ngrams(ref, char_order, word_order)):
        match = sum(min(c, r[g]) for g, c in h.items() if g in r)
        stats += [sum(h.values()) if r else 0, sum(r.values()), match]
    return stats


def chrf_from_stats(stats, beta=2):
    factor = beta ** 2
    avg_p = avg_r = 0.0
    eff = 0
    for i in range(len(stats) // 3):
        n_hyp, n_ref, n_match = stats[3 * i:3 * i + 3]
        if n_hyp > 0 and n_ref > 0:
            avg_p += n_match / n_hyp
            avg_r += n_match / n_ref
            eff += 1
    if eff == 0:
        return 0.0
    avg_p, avg_r = avg_p / eff, avg_r / eff
    if avg_p + avg_r == 0:
        return 0.0
    return 100.0 * (1 + factor) * avg_p * avg_r / (factor * avg_p + avg_r)


# ---------------------------------------------------------------- public API
def _sum_stats(rows):
    return [sum(col) for col in zip(*rows)] if rows else None


def segment_stats(preds, refs):
    """Per-segment (bleu_stats, chrf_stats); lets bootstrap resample without recomputing n-grams."""
    preds = ['' if p is None else str(p) for p in preds]
    refs = ['' if r is None else str(r) for r in refs]
    assert len(preds) == len(refs), f"{len(preds)} predictions vs {len(refs)} references"
    return [bleu_stats(p, r) for p, r in zip(preds, refs)], [chrf_stats(p, r) for p, r in zip(preds, refs)]


def scores_from_segment_stats(bleu_rows, chrf_rows):
    bleu = bleu_from_stats(_sum_stats(bleu_rows))
    chrfpp = chrf_from_stats(_sum_stats(chrf_rows))
    return {"bleu": bleu, "chrfpp": chrfpp, "geo_mean": math.sqrt(max(bleu, 0.0) * max(chrfpp, 0.0))}


def corpus_scores(preds, refs):
    """Corpus BLEU, chrF++ and their geometric mean (the competition score), all on 0-100."""
    return scores_from_segment_stats(*segment_stats(preds, refs))


def sentence_chrfpp(hyp, ref):
    """Sentence-level chrF++, used as the MBR utility."""
    return chrf_from_stats(chrf_stats(hyp, ref))


METRIC_SIGNATURE = "BLEU=sacrebleu-2.x-13a-exp | chrF++=sacrebleu-2.x-c6-w2-b2 | geo_mean=sqrt(BLEU*chrF++)"

In [ ]:
# >>> SHARED CELL: common/runlog.py
# Auto-copied from the repository by tools/sync_shared_cells.py. Do not edit here;
# identical in mbart50, byte-level-seq2seq, pipeline_QwenQLoRa and ensemble-pipeline.
import datetime
import json
import os

import pandas as pd


RESULTS_DIR = "results"


def finalize_run(system, seed, ids, refs, preds, hyperparameters, log_history=None,
                 out_dir=RESULTS_DIR, notes=None):
    os.makedirs(out_dir, exist_ok=True)
    ids = [str(i) for i in ids]
    refs = ["" if r is None else str(r) for r in refs]
    preds = ["" if p is None else str(p) for p in preds]
    scores = corpus_scores(preds, refs)
    tag = f"{system}__seed{seed}"
    pred_path = os.path.join(out_dir, f"preds_val_{tag}.csv")
    pd.DataFrame({"id": ids, "target_text": refs, "pred_translation": preds}).to_csv(pred_path, index=False)
    record = {
        "system": system,
        "seed": seed,
        "created_utc": datetime.datetime.utcnow().isoformat(timespec="seconds"),
        "metric": METRIC_SIGNATURE,
        "n_val_docs": len(ids),
        "val_ids_sha256": val_ids_sha256(ids),
        "final_val": scores,
        "hyperparameters": hyperparameters,
        "notes": notes,
        "log_history": log_history or [],
    }
    with open(os.path.join(out_dir, f"run_{tag}.json"), "w") as f:
        json.dump(record, f, indent=2, default=str, ensure_ascii=False)
    print(f"[run record] {tag}: BLEU={scores['bleu']:.2f} chrF++={scores['chrfpp']:.2f} "
          f"GM={scores['geo_mean']:.2f} -> {pred_path}")
    return scores

In [ ]:
# >>> SHARED CELL: common/env.py
# Auto-copied from the repository by tools/sync_shared_cells.py. Do not edit here;
# identical in mbart50, byte-level-seq2seq, pipeline_QwenQLoRa and ensemble-pipeline.
import importlib
import json
import os
import platform
import sys

TRACKED_PACKAGES = ["torch", "transformers", "datasets", "accelerate", "peft", "bitsandbytes",
                    "sentencepiece", "sacrebleu", "pandas", "numpy", "scikit-learn"]


def report_environment(notebook, out_dir="results"):
    versions = {}
    for name in TRACKED_PACKAGES:
        try:
            module = importlib.import_module("sklearn" if name == "scikit-learn" else name)
            versions[name] = getattr(module, "__version__", "unknown")
        except Exception:
            versions[name] = None
    env = {"notebook": notebook, "python": sys.version.split()[0], "platform": platform.platform(),
           "packages": versions, "gpu": None, "cuda": None}
    try:
        import torch
        if torch.cuda.is_available():
            env["gpu"] = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
            env["cuda"] = torch.version.cuda
    except Exception:
        pass
    os.makedirs(out_dir, exist_ok=True)
    with open(os.path.join(out_dir, f"env_{notebook}.json"), "w") as f:
        json.dump(env, f, indent=2)
    print(json.dumps(env, indent=2))
    return env

In [ ]:
report_environment("ensemble")  # -> results/env_ensemble.json

## Load the data once, up front — raw `train.csv` / `test.csv`, no normalization

The validation documents come from the shared `make_split` cell: the same 156 documents on which every member model was validated and which none of them was trained on (after the Phase-2 retraining, all three models are trained on the shared split). Neither `val_texts` nor `test_texts` get any text transformation.

In [18]:
import pandas as pd

COMP_DIR = "/kaggle/input/competitions/deep-past-initiative-machine-translation"

# Shared document-level split (identical in every notebook, verified by checksum).
_, val_df = make_split(pd.read_csv(f"{COMP_DIR}/train.csv"))
val_texts = val_df["transliteration"].tolist()   # no normalization
val_refs = val_df["translation"].tolist()

test_df = pd.read_csv(f"{COMP_DIR}/test.csv")
test_texts = test_df["transliteration"].tolist()  # no normalization

print(f"val: {len(val_texts)} documents, test: {len(test_texts)} rows")

val: 156 sentences, test: 4 sentences


## Per-model load / generate / unload

Each model gets a `load_*()` function (returns whatever `generate_*()` needs) and a `generate_*()` function (batched, returns `(texts, confidences)` — confidences are the beam-search `sequences_scores` log-probs, used later to weight votes). `load_byt5()` always resolves to the local Kaggle path; `load_mbart()`/`load_qwen()` always resolve to Hub repo ids. Nothing here loads a model yet; the loop in the next cell does that one model at a time.

In [19]:
import gc
import math
import torch
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# Per-model limits = the ones each model was trained with (see the training notebooks).
GEN_LIMITS = {  # name: (max source length, max new tokens, batch size)
    "byt5": (1024, 1408, 2),   # bytes
    "mbart": (576, 448, 4),    # mBART tokens
    "qwen": (1024, 448, 1),    # Qwen tokens (prompt incl. source)
}
NUM_BEAMS = 4


def unload(ctx=None):
    if isinstance(ctx, dict):
        ctx.clear()
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def _seq2seq_generate_batched(model, tokenizer, texts, limits, forced_bos_token_id=None):
    max_src, max_new, batch_size = limits
    all_texts, all_scores = [], []
    for i in tqdm(range(0, len(texts), batch_size), leave=False):
        chunk = texts[i:i + batch_size]
        inputs = tokenizer(
            chunk, return_tensors="pt", truncation=True,
            max_length=max_src, padding=True,
        ).to(model.device)
        gen_kwargs = {"forced_bos_token_id": forced_bos_token_id} if forced_bos_token_id is not None else {}
        with torch.no_grad():
            out = model.generate(
                **inputs,
                **gen_kwargs,
                max_new_tokens=max_new,
                num_beams=NUM_BEAMS,
                return_dict_in_generate=True,   # output_scores removed
            )
        scores = (out.sequences_scores.tolist() if getattr(out, "sequences_scores", None) is not None
                  else [0.0] * len(out.sequences))
        decoded = tokenizer.batch_decode(out.sequences, skip_special_tokens=True)
        all_texts.extend(d.strip() for d in decoded)
        all_scores.extend(scores)
    return all_texts, all_scores


# --- ByT5 (Kaggle-only checkpoint) --------------------------------------------
def load_byt5():
    tok = AutoTokenizer.from_pretrained(BYT5_REPO)
    model = AutoModelForSeq2SeqLM.from_pretrained(BYT5_REPO).to(DEVICE).eval()
    return {"model": model, "tokenizer": tok}


def generate_byt5(ctx, texts):
    return _seq2seq_generate_batched(ctx["model"], ctx["tokenizer"], texts, GEN_LIMITS["byt5"])


# --- mBART-50 (Hub) -------------------------------------------------------------
def load_mbart():
    repo = MBART_REPO if IS_ONLINE else MBART_OFFLINE_PATH
    tok = AutoTokenizer.from_pretrained(repo)
    tok.src_lang = "en_XX"
    tok.tgt_lang = "en_XX"
    model = AutoModelForSeq2SeqLM.from_pretrained(repo).to(DEVICE).eval()
    forced_bos = tok.convert_tokens_to_ids("en_XX")  # was the function itself, so it was never applied
    return {"model": model, "tokenizer": tok, "forced_bos": forced_bos}


def generate_mbart(ctx, texts):
    return _seq2seq_generate_batched(ctx["model"], ctx["tokenizer"], texts, GEN_LIMITS["mbart"], forced_bos_token_id=ctx["forced_bos"])


# --- QLoRA Qwen (Hub) ------------------------------------------------------------
# Must be EXACTLY the prompt used in training (pipeline_QwenQLoRa.ipynb, build_prompt). Before Phase 2 the
# ensemble used a different prompt than the one the adapter was trained on.
QWEN_SYSTEM_PROMPT = (
    "You are an expert translator of Akkadian. "
    "Translate the Akkadian transliteration into accurate English. "
    "Return only the English translation."
)
QWEN_PROMPT_TEMPLATE = QWEN_SYSTEM_PROMPT + "\n\nAkkadian:\n{source}\n\nEnglish:\n"


def load_qwen():
    base_repo = QWEN_BASE_REPO if IS_ONLINE else QWEN_BASE_OFFLINE_PATH
    adapter_repo = QWEN_ADAPTER_REPO if IS_ONLINE else QWEN_ADAPTER_OFFLINE_PATH

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    tok = AutoTokenizer.from_pretrained(base_repo)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "left"  # required for batched generate() with a causal LM

    base = AutoModelForCausalLM.from_pretrained(
        base_repo, quantization_config=bnb_config, device_map="auto",
    )
    model = PeftModel.from_pretrained(base, adapter_repo).eval()
    return {"model": model, "tokenizer": tok}


def generate_qwen(ctx, texts):
    model, tok = ctx["model"], ctx["tokenizer"]
    max_src, max_new, batch_size = GEN_LIMITS["qwen"]
    all_texts, all_scores = [], []
    for i in tqdm(range(0, len(texts), batch_size), leave=False):
        chunk = texts[i:i + batch_size]
        prompts = [QWEN_PROMPT_TEMPLATE.format(source=t) for t in chunk]
        inputs = tok(
            prompts, return_tensors="pt", truncation=True,
            max_length=max_src, padding=True,
        ).to(model.device)
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=max_new,
                num_beams=NUM_BEAMS,
                return_dict_in_generate=True,   # output_scores removed
            )
        scores = (out.sequences_scores.tolist() if getattr(out, "sequences_scores", None) is not None
                  else [0.0] * len(out.sequences))
        gen_only = out.sequences[:, inputs["input_ids"].shape[1]:]  # strip the prompt back off
        decoded = tok.batch_decode(gen_only, skip_special_tokens=True)
        all_texts.extend(d.strip() for d in decoded)
        all_scores.extend(scores)
    return all_texts, all_scores


MODEL_SPECS = {
    "byt5": {"load": load_byt5, "generate": generate_byt5},
    "mbart": {"load": load_mbart, "generate": generate_mbart},
    "qwen": {"load": load_qwen, "generate": generate_qwen},
}

## Sequential run — one model in memory at a time

For each model: load it, translate the *entire* val set and the *entire* test set, stash the results, then unload it before moving to the next. `predictions[model_name]["val"|"test"]` holds `(texts, confidences)` afterwards; no model weights are kept around once this cell finishes.

In [20]:
seed_everything()  # re-seed before generation

predictions = {}

for name, spec in MODEL_SPECS.items():
    print(f"\n=== Loading {name} ===")
    ctx = spec["load"]()

    print(f"Translating validation set with {name}...")
    val_texts_out, val_scores_out = spec["generate"](ctx, val_texts)

    print(f"Translating test set with {name}...")
    test_texts_out, test_scores_out = spec["generate"](ctx, test_texts)

    predictions[name] = {
        "val": (val_texts_out, val_scores_out),
        "test": (test_texts_out, test_scores_out),
    }

    print(f"Unloading {name}...")
    del val_texts_out, val_scores_out, test_texts_out, test_scores_out
    unload(ctx)
    del ctx
    gc.collect(); torch.cuda.empty_cache()
    print(torch.cuda.memory_allocated() / 1e9, "GB still allocated")

print("\nAll three models translated and unloaded. Peak GPU memory was one model at a time.")



=== Loading byt5 ===


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Translating validation set with byt5...


KeyboardInterrupt: 

## Orchestration algorithm — Minimum Bayes Risk consensus

For each sentence, all three models proposed a candidate (already generated and stored above). Instead of a fixed rule ("always trust mBART"), the orchestrator scores each candidate by its **chrF similarity to the other two candidates**, weighted by each model's decoding confidence (and an optional tunable prior per model). The candidate that best "agrees" with the others wins.

This works because the three models are structurally diverse (byte-level seq2seq, LoRA-tuned decoder-only LM, denoising seq2seq) — they're unlikely to make the *same* mistake by chance, so agreement is a reasonable proxy for correctness even without a reference translation at inference time. It degrades gracefully: if one model fails to produce output, MBR falls back to whichever of the remaining candidates agrees most; if only one model produces output, it's used as-is.

Tune `MODEL_PRIORS` from the validation results in the next section if one model turns out systematically stronger.

In [ ]:
from collections import Counter

def _char_ngrams(s, n):
    s = s.replace(" ", "")
    return Counter(s[i:i + n] for i in range(len(s) - n + 1)) if len(s) >= n else Counter()


def sentence_chrf(a, b, max_n=6, beta=2):
    """Sentence-level chrF (character n-gram F-score), used as the MBR utility function."""
    if not a or not b:
        return 0.0
    ps, rs = [], []
    for n in range(1, max_n + 1):
        ca, cb = _char_ngrams(a, n), _char_ngrams(b, n)
        overlap = sum((ca & cb).values())
        ps.append(overlap / max(sum(ca.values()), 1))
        rs.append(overlap / max(sum(cb.values()), 1))
    p, r = sum(ps) / max_n, sum(rs) / max_n
    if p + r == 0:
        return 0.0
    beta2 = beta ** 2
    return (1 + beta2) * p * r / (beta2 * p + r)


# Relative trust per model — start at 1.0 each; adjust after looking at per-model
# validation chrF/BLEU below (e.g. downweight a model that's consistently worse).
MODEL_PRIORS = {"byt5": 1.0, "mbart": 1.0, "qwen": 1.0}


def mbr_orchestrate(candidates):
    """
    candidates: list of {"model": str, "text": str, "conf": float}  (conf = generate()'s
    sequence log-prob, <= 0; more negative = less confident)

    Returns (winner_dict, all_candidates_with_utility) — winner_dict has an extra
    "utility" key so you can inspect how confident the consensus was.
    """
    valid = [c for c in candidates if c["text"]]
    if not valid:
        return {"model": None, "text": "", "utility": 0.0}, []
    if len(valid) == 1:
        only = {**valid[0], "utility": 1.0}
        return only, [only]

    weights = []
    for c in valid:
        prior = MODEL_PRIORS.get(c["model"], 1.0)
        conf = math.exp(c["conf"]) if c["conf"] <= 0 else c["conf"]  # log-prob -> (0, 1]
        weights.append(prior * conf)
    total_w = sum(weights) or 1.0
    weights = [w / total_w for w in weights]

    scored = []
    for i, ci in enumerate(valid):
        utility = sum(
            weights[j] * sentence_chrf(ci["text"], cj["text"])
            for j, cj in enumerate(valid) if j != i
        )
        scored.append({**ci, "utility": utility})

    winner = max(scored, key=lambda c: c["utility"])
    return winner, scored


def orchestrate_all(split):
    """split: 'val' or 'test'. Reads from the `predictions` dict filled in by the sequential run."""
    n = len(predictions["byt5"][split][0])
    finals, log = [], []
    for i in range(n):
        candidates = [
            {"model": name, "text": predictions[name][split][0][i], "conf": predictions[name][split][1][i]}
            for name in MODEL_SPECS
        ]
        winner, scored = mbr_orchestrate(candidates)
        finals.append(winner["text"])
        log.append(scored)
    return finals, log


## Validation — ensemble vs. each individual model

Scores come from the shared metrics cell (sacreBLEU-identical BLEU and chrF++, no internet needed). Every member and the ensemble also get a run record in `results/` via `finalize_run`. MLflow logging of `val_bleu`, `val_chrf_pp`, `val_geo_mean_chrf_bleu` (and a `val_avg_logprob` confidence proxy) happens only when `EXECUTION_MODE == "online"`.

In [ ]:
import numpy as np
# Corpus BLEU / chrF++ come from the shared metrics cell (`corpus_scores`).

In [ ]:
ensemble_val, ensemble_val_log = orchestrate_all("val")

MEMBER_CHECKPOINTS = {
    "byt5": BYT5_REPO,
    "mbart": MBART_REPO if IS_ONLINE else MBART_OFFLINE_PATH,
    "qwen": QWEN_ADAPTER_REPO if IS_ONLINE else QWEN_ADAPTER_OFFLINE_PATH,
}

def _avg_logprob(scores):
    scores = [s for s in scores if s is not None]
    return sum(scores) / len(scores) if scores else None

print(f"{'system':10s}  {'BLEU':>6s}  {'chrF++':>7s}  {'geo_mean':>8s}")

mlflow_parent_run = mlflow.start_run(run_name="ensemble-inference") if IS_ONLINE else None
parent_metrics = {}

per_model_metrics = {}
for name in MODEL_SPECS:
    preds, scores = predictions[name]["val"]
    _s = finalize_run(system=f"ensemble-member-{name}", seed=SEED, ids=val_df["oare_id"], refs=val_refs,
                      preds=preds, hyperparameters={"checkpoint": MEMBER_CHECKPOINTS[name]})
    bleu, chrf_pp, geo_mean = _s["bleu"], _s["chrfpp"], _s["geo_mean"]
    per_model_metrics[name] = {"val_bleu": bleu, "val_chrf_pp": chrf_pp, "val_geo_mean_chrf_bleu": geo_mean}
    print(f"{name:10s}  {bleu:6.2f}  {chrf_pp:7.2f}  {geo_mean:8.2f}")

    log_dict = dict(per_model_metrics[name])
    avg_lp = _avg_logprob(scores)
    if avg_lp is not None:
        log_dict["val_avg_logprob"] = avg_lp  # rough confidence proxy, not a true loss

    # Log every metric on the per-model nested run AND, prefixed, on the parent run —
    # so metrics show up whichever run entry gets checked on DagsHub.
    if IS_ONLINE:
        with mlflow.start_run(run_name=f"val-{name}", nested=True):
            mlflow.set_tag("model", name)
            mlflow.log_metrics(log_dict)
    for k, v in log_dict.items():
        parent_metrics[f"{name}_{k}"] = v

_s = finalize_run(system="mbr3-ensemble", seed=SEED, ids=val_df["oare_id"], refs=val_refs, preds=ensemble_val,
                  hyperparameters={"members": MEMBER_CHECKPOINTS, "priors": MODEL_PRIORS,
                                   "utility": "sentence chrF (char 6-grams), confidence-weighted"})
ensemble_bleu, ensemble_chrf_pp, ensemble_geo_mean = _s["bleu"], _s["chrfpp"], _s["geo_mean"]
print(f"{'ensemble':10s}  {ensemble_bleu:6.2f}  {ensemble_chrf_pp:7.2f}  {ensemble_geo_mean:8.2f}")

ensemble_metrics = {
    "val_bleu": ensemble_bleu,
    "val_chrf_pp": ensemble_chrf_pp,
    "val_geo_mean_chrf_bleu": ensemble_geo_mean,
}
if IS_ONLINE:
    with mlflow.start_run(run_name="val-ensemble", nested=True):
        mlflow.set_tag("model", "ensemble")
        mlflow.log_metrics(ensemble_metrics)
for k, v in ensemble_metrics.items():
    parent_metrics[f"ensemble_{k}"] = v

winner_counts = Counter(max(row, key=lambda c: c['utility'])['model'] for row in ensemble_val_log)
print("\nWinning model per sentence:", dict(winner_counts))

if IS_ONLINE:
    # Log the full combined set (every model + ensemble) directly on the parent run too.
    mlflow.log_metrics(parent_metrics)
    mlflow.log_metrics({f"winner_count_{k}": v for k, v in winner_counts.items()})
    mlflow.end_run()  # closes mlflow_parent_run

## Test-set inference (submission.csv)

Also writes `ensemble_candidates_debug.csv` with every model's raw candidate + which one the orchestrator picked, for auditing. No model loading or network access happens here — it's all reading from `predictions`, already populated by the sequential run above.

In [ ]:
final_translations, test_log = orchestrate_all("test")

submission = pd.DataFrame({"id": test_df["id"], "translation": final_translations})
submission.to_csv("submission.csv", index=False)
print("Saved submission.csv")

debug_rows = []
for row_id, scored_row in zip(test_df["id"], test_log):
    entry = {"id": row_id}
    for c in scored_row:
        entry[f"{c['model']}_text"] = c["text"]
        entry[f"{c['model']}_utility"] = c["utility"]
    entry["chosen_model"] = max(scored_row, key=lambda c: c["utility"])["model"]
    debug_rows.append(entry)
pd.DataFrame(debug_rows).to_csv("ensemble_candidates_debug.csv", index=False)
print("Saved ensemble_candidates_debug.csv")
